In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. 准备数据 (两个版本完全一样)
# ==========================================
torch.manual_seed(42)
help(torch.manual_seed)  # 查看函数说明
X = torch.rand(100, 1) * 10  # 100个数据，0-10之间
y = 2 * X + 3 + torch.randn(100, 1) * 0.5  # 真实关系 y=2x+3，加一点噪音

# ==========================================
# 2. 定义模型零件 (散装！)
# ==========================================
# 我们得手动创建每一层，作为独立的变量
# 模型结构: 输入(1) -> 隐藏层(10) -> 输出(1)
layer1 = nn.Linear(1, 10)  # 第一层：1个输入，10个神经元
activation = nn.ReLU()      # 激活函数
layer2 = nn.Linear(10, 1)  # 第二层：10个输入，1个输出

# ==========================================
# 3. 手动写前向传播函数
# ==========================================
# 每次都要把数据手动传给每一层
def manual_forward(x):
    x = layer1(x)
    x = activation(x)
    x = layer2(x)
    return x



Help on function manual_seed in module torch.random:

manual_seed(seed) -> torch._C.Generator
    Sets the seed for generating random numbers on all devices. Returns a
    `torch.Generator` object.
    
    Args:
        seed (int): The desired seed. Value must be within the inclusive range
            `[-0x8000_0000_0000_0000, 0xffff_ffff_ffff_ffff]`. Otherwise, a RuntimeError
            is raised. Negative inputs are remapped to positive values with the formula
            `0xffff_ffff_ffff_ffff + seed`.



In [10]:
# ==========================================
# 4. 定义损失和优化器 
# ==========================================
criterion = nn.MSELoss()

# 优化器这里很麻烦：你必须手动把所有层的参数一个一个列出来传给它
print({'weights': layer1.weight})  # 这是第一层的权重参数
print({'bias': layer1.bias})  # 这是第一层的偏置参数
# 如果有10层，你就要列10次...
optimizer = optim.SGD([
    {'params': layer1.parameters()},
    {'params': layer2.parameters()}
], lr=0.01)

# ==========================================
# 5. 训练循环
# ==========================================
epochs = 1000
for epoch in range(epochs):
    # 前向传播：调用手动写的函数
    predictions = manual_forward(X)
    loss = criterion(predictions, y)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# ==========================================
# 6. 查看结果
# ==========================================
# 注意：因为加了隐藏层和ReLU，我们没法直接看 w=2, b=3 了，看Loss就行
print(f"\n训练完成！最终Loss: {loss.item():.4f}")

{'weights': Parameter containing:
tensor([[-1.2033],
        [-0.5368],
        [-0.1222],
        [-0.3809],
        [-0.6473],
        [-3.1850],
        [-0.2409],
        [-0.8331],
        [-0.9925],
        [-2.2601]], requires_grad=True)}
{'bias': Parameter containing:
tensor([ 0.2428,  5.1110, -0.7762, -0.9869, -0.8122,  0.1287,  2.2922,  7.8767,
         0.5518, -0.1709], requires_grad=True)}
Epoch [200/1000], Loss: 0.5515
Epoch [400/1000], Loss: 0.5664
Epoch [600/1000], Loss: 0.5767
Epoch [800/1000], Loss: 0.5318
Epoch [1000/1000], Loss: 0.4595

训练完成！最终Loss: 0.4595


# 我们不希望散装的创建每一层，也不希望手动写传入参数

In [ ]:
# CLASS
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. 准备数据 (和上面完全一样)
# ==========================================
torch.manual_seed(42)
X = torch.rand(100, 1) * 10
y = 2 * X + 3 + torch.randn(100, 1) * 0.5

# ==========================================
# 2. 定义模型 (封装！)
# ==========================================
class TwoLayerNet(nn.Module):
    def __init__(self):
        super(TwoLayerNet, self).__init__() # 先调用父类的初始化方法，父类是 nn.Module
        # 把所有零件都装在这个“盒子”里
        self.layer1 = nn.Linear(1, 10)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(10, 1)

    def forward(self, x):
        # 数据怎么流，在这里写清楚
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

# ==========================================
# 3. 实例化模型 (像变魔术一样！)
# ==========================================
model = TwoLayerNet()

# ==========================================
# 4. 定义损失和优化器 
# ==========================================
criterion = nn.MSELoss()

# 优化器这里只需要传 model.parameters()
# PyTorch 自动帮你把盒子里所有的 layer1, layer2 都找出来了！
optimizer = optim.SGD(model.parameters(), lr=0.01)

# ==========================================
# 5. 训练循环
# ==========================================
epochs = 1000
print("--- 使用 Class 开始训练 ---")
for epoch in range(epochs):
    # 前向传播：直接调用 model(x)，就像调用函数一样
    predictions = model(X)
    loss = criterion(predictions, y)
    
    # 反向传播 (一模一样)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# ==========================================
# 6. 查看结果 (简单)
# ==========================================
print(f"\n训练完成！最终Loss: {loss.item():.4f}")

# 额外福利：你还可以一键保存/加载模型
# torch.save(model.state_dict(), 'my_model.pth') # 保存

--- 使用 Class 开始训练 ---
Epoch [200/1000], Loss: 9.6160
Epoch [400/1000], Loss: 3.0057
Epoch [600/1000], Loss: 8.0201
Epoch [800/1000], Loss: 0.9779
Epoch [1000/1000], Loss: 1.9927

训练完成！最终Loss: 1.9927
